# 🎯 Smart Resume Keyword Matcher

> **Portfolio Project** | Classical NLP + TF-IDF + Cosine Similarity
>
> Compares a Resume PDF against a Job Description and returns:
> - An overall **match score (%)**
> - A **section breakdown** (Skills / Experience / Education)
> - A prioritized list of **missing keywords**
> - A **history log** of all past comparisons

---

**Tech Stack:** `pdfplumber` · `spaCy (en_core_web_sm)` · `scikit-learn (TF-IDF)` · `Gradio`

**Setup:**
```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```

## ⚙️ Setup & Imports

In [ ]:
# ── Setup Verification ────────────────────────────────────────────────────────
# Run this cell FIRST to verify all dependencies are installed correctly.
# If any import fails, run: pip install -r requirements.txt
# Then run: python -m spacy download en_core_web_sm

import sys
import importlib
import os

# Check all required packages are importable
REQUIRED = ["pdfplumber", "spacy", "sklearn", "gradio", "pandas", "numpy"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"❌ Missing packages: {missing}")
    print("   Fix: pip install -r requirements.txt")
else:
    print("✅ All dependencies found. You're offline-ready.")

# Load spaCy model — works fully offline after the first download
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy model 'en_core_web_sm' loaded.")
except OSError:
    print("❌ spaCy model not found.")
    print("   Fix: python -m spacy download en_core_web_sm")

# Verify history.json exists — create it if missing
HISTORY_PATH = "history.json"
if not os.path.exists(HISTORY_PATH):
    import json
    with open(HISTORY_PATH, "w") as f:
        json.dump([], f)
    print(f"✅ Created {HISTORY_PATH}")
else:
    print(f"✅ {HISTORY_PATH} found.")

print("\n🚀 Setup complete — ready to match resumes!")

## 📄 Phase 1: PDF Parsing & Section Detection

These two functions form the **input pipeline**:
1. `extract_text_from_pdf()` — reads a PDF and returns raw text
2. `detect_sections()` — splits resume text into Skills / Experience / Education buckets

Both functions are **offline** and require no internet at runtime.

In [ ]:
# ── PDF Text Extractor ────────────────────────────────────────────────────────
# Uses pdfplumber to extract raw text from every page of a resume PDF.
# pdfplumber handles multi-column layouts better than PyPDF2 — which matters
# because many modern resume templates use a two-column format.

import pdfplumber
import re

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract all text from a PDF file, page by page.

    Args:
        pdf_path: Absolute or relative path to the PDF file.

    Returns:
        A single cleaned string of all text from the PDF.
        Returns empty string if extraction fails (logs error to stdout).
    """
    full_text = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                page_text = page.extract_text()

                # Some pages may return None — e.g., image-only scanned pages
                if page_text:
                    full_text.append(page_text)
                else:
                    print(f"  ⚠️ Page {page_num}: No text extracted (may be image-based)")

    except FileNotFoundError:
        # Don't crash — return empty string and let the caller handle it gracefully
        print(f"❌ File not found: {pdf_path}")
        return ""
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        return ""

    # Join all pages, then clean up messy whitespace
    combined = "\n".join(full_text)
    combined = re.sub(r'\n{3,}', '\n\n', combined)   # max 2 consecutive blank lines
    combined = re.sub(r'[ \t]{2,}', ' ', combined)   # collapse inline multiple spaces

    return combined.strip()


# ── Quick Smoke Test ──────────────────────────────────────────────────────────
# Place a PDF in sample_resumes/ and uncomment these lines to test:
# sample_text = extract_text_from_pdf("sample_resumes/test_resume.pdf")
# print(f"Extracted {len(sample_text)} characters")
# print(sample_text[:500])  # Preview first 500 chars

In [ ]:
# ── Section Detector ─────────────────────────────────────────────────────────
# Heuristically splits a resume's raw text into three sections:
# Skills, Experience, and Education.
#
# Strategy: scan line by line for section-header keywords.
# This is fast, offline, and 'good enough' for classical NLP scoring.
# Imperfect splits are smoothed out by cosine similarity averaging in Phase 2.

def detect_sections(text: str) -> dict:
    """
    Split resume text into Skills, Experience, Education, and General sections.

    Uses header-keyword detection to find section boundaries. Text between
    boundaries is assigned to that section. Unclassified text goes to 'general'.

    Args:
        text: Raw resume text (output of extract_text_from_pdf).

    Returns:
        dict with keys: 'skills', 'experience', 'education', 'general'
        Each value is a string of text belonging to that section.
    """
    if not text:
        # Handle empty input gracefully — return empty sections
        return {"skills": "", "experience": "", "education": "", "general": ""}

    # Keywords that signal the start of each section (case-insensitive)
    SECTION_KEYWORDS = {
        "skills": [
            "skill", "technical skill", "core competenc", "technologies",
            "tools", "programming", "languages", "proficienc"
        ],
        "experience": [
            "experience", "work history", "employment", "career",
            "professional background", "work experience", "internship"
        ],
        "education": [
            "education", "academic", "qualification", "degree",
            "university", "college", "schooling", "certification"
        ],
    }

    sections = {"skills": [], "experience": [], "education": [], "general": []}
    current_section = "general"  # Default — text before any known header

    for line in text.split("\n"):
        line_lower = line.lower().strip()

        # Only check for section change if this line is short enough to be a header.
        # The 40-char guard prevents long sentences that happen to contain keywords
        # (e.g., "5 years of Python experience") from being misclassified as headers.
        matched_section = None
        if len(line_lower) <= 40:
            for section, keywords in SECTION_KEYWORDS.items():
                if any(kw in line_lower for kw in keywords):
                    matched_section = section
                    break

        if matched_section:
            current_section = matched_section
        else:
            sections[current_section].append(line)

    # Join the accumulated lines back into strings
    return {k: "\n".join(v).strip() for k, v in sections.items()}


# ── Quick Smoke Test ──────────────────────────────────────────────────────────
# sample_sections = detect_sections(sample_text)
# for section, content in sample_sections.items():
#     preview = content[:150] if content else "(empty)"
#     print(f"\n── {section.upper()} ──\n{preview}")

## 🧠 Phase 2: NLP Scoring

Four functions that form the **scoring and analysis pipeline**:
1. `preprocess_text()` — lemmatize and clean text for TF-IDF input
2. `compute_match_score()` — TF-IDF + cosine similarity → 4 scores
3. `find_missing_keywords()` — JD terms absent from resume, ranked by importance
4. `format_score_output()` — data contract used by History (Phase 3) and UI (Phase 4)

In [ ]:
# ── Text Preprocessor ────────────────────────────────────────────────────────
# Before TF-IDF can compare texts, we need to normalize them.
# Raw resume text is messy: mixed case, punctuation, stopwords like "and/the/is".
# spaCy's lemmatizer reduces words to their root forms so "managing" and
# "managed" are treated as the same signal — critical for semantic matching.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def preprocess_text(text: str) -> str:
    """
    Clean and lemmatize text using spaCy for TF-IDF input.

    Steps:
    1. Lowercase everything
    2. Remove non-alphabetic tokens (numbers, punctuation, symbols)
    3. Remove spaCy stopwords (common words that add no semantic signal)
    4. Lemmatize remaining tokens (run→run, managing→manage, databases→database)

    Args:
        text: Raw text string (resume section or job description).

    Returns:
        A single space-joined string of cleaned, lemmatized tokens.
        Returns empty string if input is empty.
    """
    if not text or not text.strip():
        return ""

    # nlp is already loaded in the Setup cell (en_core_web_sm).
    # Disabling parser and NER since we only need the tokenizer + lemmatizer.
    doc = nlp(text.lower(), disable=["parser", "ner"])

    clean_tokens = [
        token.lemma_
        for token in doc
        if token.is_alpha          # drop numbers, punctuation, symbols
        and not token.is_stop      # drop "the", "and", "is", etc.
        and len(token.lemma_) > 2  # drop very short tokens like "ok", "mr"
    ]

    return " ".join(clean_tokens)


# ── Smoke Test ────────────────────────────────────────────────────────────────
# test_input = "Managing 5 databases and developing Python APIs for the team."
# print(preprocess_text(test_input))
# Expected: something like "manage database develop python api team"

In [ ]:
# ── Match Scorer ──────────────────────────────────────────────────────────────
# TF-IDF converts text into numeric vectors. Cosine similarity measures the angle
# between the resume and JD vectors — score of 1.0 = identical, 0.0 = no overlap.
#
# We compute FOUR scores:
#   1. Overall score    — whole resume vs whole JD
#   2. Skills score     — resume skills section vs JD
#   3. Experience score — resume experience section vs JD
#   4. Education score  — resume education section vs JD
#
# Why compare each section against the FULL JD?
# JDs don't have sections — they're one blob. Matching each resume section against
# the whole JD finds the relevant signal in each part.

def compute_match_score(resume_text: str, jd_text: str,
                         resume_sections: dict) -> dict:
    """
    Compute TF-IDF cosine similarity between a resume and job description.

    Args:
        resume_text: Full resume text (all sections combined).
        jd_text: Full job description text.
        resume_sections: Dict from detect_sections() — keys: skills, experience,
                         education, general.

    Returns:
        dict with keys: overall, skills, experience, education
        All values are floats between 0.0 and 100.0 (percentage).
    """
    def _score_pair(text_a: str, text_b: str) -> float:
        """Compute cosine similarity (as %) between two text strings."""
        clean_a = preprocess_text(text_a)
        clean_b = preprocess_text(text_b)

        # Nothing to compare if either side is empty after cleaning
        if not clean_a or not clean_b:
            return 0.0

        # Fit TF-IDF on both texts together so they share the same vocabulary
        vectorizer = TfidfVectorizer()
        try:
            tfidf_matrix = vectorizer.fit_transform([clean_a, clean_b])
        except ValueError:
            # Raised when vocabulary is empty after stop-word filtering
            return 0.0

        # [0][1] = similarity between doc 0 (resume) and doc 1 (JD)
        score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(float(score) * 100, 1)   # → percentage with 1 decimal

    return {
        "overall":    _score_pair(resume_text, jd_text),
        "skills":     _score_pair(resume_sections.get("skills",     ""), jd_text),
        "experience": _score_pair(resume_sections.get("experience", ""), jd_text),
        "education":  _score_pair(resume_sections.get("education",  ""), jd_text),
    }


# ── Smoke Test ────────────────────────────────────────────────────────────────
# test_resume   = "Python developer with 3 years of machine learning and SQL databases."
# test_jd       = "Looking for a Python engineer skilled in data science and SQL."
# test_sections = detect_sections(test_resume)
# scores = compute_match_score(test_resume, test_jd, test_sections)
# print(scores)   # Expected: overall > 0, all values between 0 and 100

In [ ]:
# ── Missing Keyword Extractor ─────────────────────────────────────────────────
# Answers: "What does the JD care about that my resume doesn't mention?"
#
# Strategy: fit TF-IDF on the JD alone to get its vocabulary weighted by
# importance. Then filter to only terms NOT found in the resume.
# TF-IDF weighting is smarter than simple set-difference — common words like
# "work" naturally score low, while distinctive terms like "kubernetes" score high.

def find_missing_keywords(resume_text: str, jd_text: str,
                            top_n: int = 15) -> list:
    """
    Find the most important JD keywords that are missing from the resume.

    Args:
        resume_text: Full resume text (raw — preprocessing happens internally).
        jd_text: Full job description text.
        top_n: Max number of missing keywords to return (default: 15).

    Returns:
        List of strings (missing keywords), ranked by TF-IDF importance.
        Returns empty list if no keywords are missing or on any error.
    """
    if not resume_text or not jd_text:
        return []

    # Preprocess both texts the same way we did for scoring
    clean_resume = preprocess_text(resume_text)
    clean_jd     = preprocess_text(jd_text)

    if not clean_jd:
        return []

    # Build a set of all tokens in the resume for fast O(1) membership check
    resume_tokens = set(clean_resume.split())

    # Fit TF-IDF on just the JD to rank its own vocabulary by importance
    vectorizer = TfidfVectorizer()
    try:
        vectorizer.fit([clean_jd])
    except ValueError:
        return []

    # Extract all JD terms with their weights, sorted highest-first
    feature_names = vectorizer.get_feature_names_out()
    tfidf_scores  = vectorizer.transform([clean_jd]).toarray()[0]

    jd_terms_ranked = sorted(
        zip(feature_names, tfidf_scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Only return JD terms that are genuinely absent from the resume
    missing = [
        term for term, score in jd_terms_ranked
        if term not in resume_tokens
    ]

    return missing[:top_n]


# ── Smoke Test ────────────────────────────────────────────────────────────────
# test_resume = "Python developer with experience in SQL and data analysis."
# test_jd = "Seeking a machine learning engineer with Kubernetes, Docker, and TensorFlow skills."
# missing = find_missing_keywords(test_resume, test_jd, top_n=10)
# print("Missing keywords:", missing)
# Expected: ['kubernetes', 'docker', 'tensorflow', 'machine', 'learning', ...]

In [ ]:
# ── Score Output Formatter ────────────────────────────────────────────────────
# Packages all NLP outputs into ONE standardized dict.
#
# This function is the DATA CONTRACT between:
#   - Phase 3 (history saver) — writes this dict to history.json
#   - Phase 4 (Gradio UI)    — reads this dict to render scores and keywords
#
# Both phases depend on the exact key names here. Change a key name and
# you must update Phase 3 and Phase 4 accordingly.

from datetime import datetime

def format_score_output(resume_filename: str,
                         jd_snippet: str,
                         scores: dict,
                         missing_keywords: list) -> dict:
    """
    Package NLP results into a standardized output dict.

    Args:
        resume_filename: Name of the uploaded resume PDF (for history logging).
        jd_snippet: First 200 chars of the JD (for history preview).
        scores: Output of compute_match_score() — 4 float keys.
        missing_keywords: Output of find_missing_keywords() — list of strings.

    Returns:
        dict with keys: timestamp, resume_filename, jd_snippet,
                        overall_score, skills_score, experience_score,
                        education_score, missing_keywords
    """
    return {
        "timestamp":        datetime.now().isoformat(timespec="seconds"),
        "resume_filename":  resume_filename,
        "jd_snippet":       jd_snippet[:200].strip(),  # keep history.json lean
        "overall_score":    scores.get("overall",    0.0),
        "skills_score":     scores.get("skills",     0.0),
        "experience_score": scores.get("experience", 0.0),
        "education_score":  scores.get("education",  0.0),
        "missing_keywords": missing_keywords,
    }


# ── Smoke Test ────────────────────────────────────────────────────────────────
# sample_result = format_score_output(
#     resume_filename="my_resume.pdf",
#     jd_snippet="Looking for a Python engineer with 3+ years of experience...",
#     scores={"overall": 72.5, "skills": 65.0, "experience": 80.0, "education": 45.0},
#     missing_keywords=["kubernetes", "docker", "tensorflow"]
# )
# import json; print(json.dumps(sample_result, indent=2))

## 💾 Phase 3: History & Persistence

*(Implementation in Phase 3 — Plan 3.x)*

Will implement:
- `save_result()` — append `format_score_output()` dict to `history.json`
- `load_history()` — read all past comparisons as a list of dicts

## 🖥️ Phase 4: Gradio UI

*(Implementation in Phase 4 — Plan 4.x)*

Will implement:
- PDF upload + JD text area + Analyze button
- Output panel: score gauge, section breakdown table, missing keywords
- History tab: past comparisons from `history.json`